# grid_refine, step by step: levels 0-1-2, spirals only, no tip jumps

The workflow that made the **good grid_refine dataset** (2026-11-03, tag `20261103_135746`),
from `selfdriving_afm/stepwise.py` (`StepwiseRun`). One cell per stage: look at each level's
figures before running the next.

- Every scan is **one PyScanner do_scan** of a pre-defined spiral (`spiral_return_opt 1`: out and
  smoothly back in) at 1 MS/s (`data_rate 0`). No PyScanner fast raster (its slow axis steps at
  every line), no Lissajous.
- **No jumps**: the backend's no-jump guard reads `get_AO0_V/AO1_V` (lagging readback, polled)
  before and after every output and drives smoothly to every start point. Every stage prints the
  guard log. With no jumps the sample did not drift (2026-11-03).
- Range 28 x 28 um; history keeps block means (`store_every 4`) so RAM stays safe.
- Each stage engages for its scans, then withdraws and drives smoothly back to (0, 0).

Settings (class attributes of `StepwiseRun`, change them in cell 1 before `setup()`):
`SPARSE` (r 7.15 V, 6 turns, 10 s), `L1_MAX_S` 4 s, `L2_MAX_S` 11.5 s, `V` 30 V/s, `Z_OBJ_NM` 30 nm,
`L2_PAD_V` 0.35 V, `PRIORS`. The rest of the configuration comes from `Run_SelfDriving_Scan.ipynb`'s
config cell, then `StepwiseRun.configure()`.

Results go to `scan_data/`: `stepwise_L2_<tag>.npz` (every scan, commanded + sensor positions, raw and
levelled Z, binned images, linear / CS / TV reconstructions, guard log) + PNGs + GIFs.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from IPython.display import Image, display
from selfdriving_afm.stepwise import StepwiseRun

run = StepwiseRun(out_dir="scan_data")
# e.g. run.SPARSE = dict(center=(0.3, 0.0), radius_V=7.15, n_cycles=6, duration_s=10.0)
#      run.L2_MAX_S = 11.5

_shown = []
def show_new(paths=None):
    """Display the figures made since the last call."""
    paths = paths if paths is not None else (run.st.figs if hasattr(run, "st") else [])
    for p in paths:
        if p not in _shown:
            display(Image(filename=str(p))); _shown.append(p)

## 1. Setup
Connects PyScanner (`hardware_paths.json`) and Igor, configures, then the Asylum setup: routing,
sensor calibration with the tip withdrawn, engage.

In [ ]:
run.setup()

## 2. Level 0: the sparse spiral (one do_scan, ~10 s)

In [ ]:
run.level0()
show_new()

## 3. Level 1: one spiral per object found on the sparse arcs (<= 4 s each)
Look at the level-0 figure first. `run.objects_from_sparse()` shows the boxes without moving anything.

In [ ]:
run.level1()
show_new()

## 4. Level 2: one dense spiral per object (<= 11.5 s each)
Placed from level 1's measured data: level 1's real-vs-commanded size and centre offset are corrected.

In [ ]:
run.level2()
show_new()

## 5. Save the dataset, figures, analysis animation

In [ ]:
run.save_dataset()
show_new(run.figures())
run.animate()
print("GIF:", run.GIF)

## 6. The equivalent raster: only when you decide
Same number of pixels, user-defined smooth waveforms (not PyScanner's fast raster): every band starts
and ends at rest, rounded turnarounds, driven moves between bands. `plan_raster()` only plans and checks
the RAM and the range; nothing moves.

In [ ]:
RP, ok = run.plan_raster()

In [ ]:
run.equivalent_raster()          # moves the tip: run only when you decide; added to the same npz

In [ ]:
show_new(run.compare_raster())   # zoom_um=(x0, x1, y0, y1) adds a zoomed column
print("GIF:", run.GIF_VS)

## (Optional) dry run on a simulated sample
Nothing is connected: `setup_simulation(topography_fn)` instead of `setup()`, then the same cells
(`topography_fn(x_V, y_V)` -> height in nm). Use a fresh `StepwiseRun`.

In [ ]:
# from selfdriving_afm.backends.simulate_backend import default_topography
# run = StepwiseRun(out_dir="scan_data/sim")
# run.setup_simulation(default_topography, sample_rate_hz=1e5)